# ⚔️ Algorithm Comparison — 6 Algorithms Head-to-Head

CircuitKit supports multiple circuit discovery algorithms, each with different trade-offs in cost, circuit quality, and data requirements.

This notebook runs all **6 stable algorithms** on the same model and task, then compares the resulting circuits using faithfulness metrics and visualizations.

| Algorithm | Type | Pairs Required? | Level | Cost |
|-----------|------|-----------------|-------|------|
| EAP | Edge Attribution Patching | Yes | node/neuron | Low |
| EAP-IG | EAP + Integrated Gradients | Yes | node/neuron | Medium |
| EAP-GP | EAP + Gradient × Path | Yes | node/neuron | Medium |
| ACDC | Automatic Circuit Discovery | Yes | node/neuron | High |
| CD-T | Causal Discovery with Tests | Yes | **node only** | Medium |
| IBCircuit | Information Bottleneck | **No** (clean-only) | node/neuron | Medium |

- **Model:** `google/gemma-2-2b-it` (~2B params, instruction-tuned)
- **Task:** IOI
- **Runtime:** ~20-30 min on Colab T4

---

## Setup

Requires GPU. Run the GPU track cells from `00_colab_setup.ipynb` first.

In [ ]:
# ⚠️ Must be set before any CUDA context is created.
# If you've already run GPU cells in a prior notebook, restart the runtime first.
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

import warnings
warnings.filterwarnings('ignore', module='transformer_lens')
warnings.filterwarnings('ignore', module='lm_eval')
warnings.filterwarnings('ignore', message='.*pretrained.*model kwarg.*')

import gc
import time
import torch
import pandas as pd
import circuitkit as ck
from circuitkit import Pipeline

print(f"CUDA: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

## Define the Experiment

We compare 6 algorithms on the same model/task configuration. CD-T is forced to node-level (it doesn't support neuron-level). IBCircuit doesn't require paired (clean/corrupt) data.

In [ ]:
MODEL = "google/gemma-2-2b-it"
TASK = "ioi"
LEVEL = "node"
SPARSITY = 0.3
N_EXAMPLES = 256  # reduce to 64 for a quick dry run
BATCH_SIZE = 2
OUTPUT_DIR = "./results/algo_comparison"

# Ordered cheapest-first so acdc (high cost) doesn't block cheaper algorithms on OOM
ALGORITHMS = ["eap", "eap-ig", "eap-gp", "cdt", "ibcircuit", "acdc"]

## Run Discovery for All 6 Algorithms

Each algorithm produces a `Circuit` with importance scores.

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

results = {}  # algo_name -> {circuit, time_s, pipe}

for algo in ALGORITHMS:
    print(f"\n{'='*60}")
    print(f"Running: {algo}")
    print(f"{'='*60}")

    pipe = Pipeline(
        MODEL, task=TASK,
        output_dir=os.path.join(OUTPUT_DIR, algo),
    )

    t0 = time.time()
    try:
        extra_kw = {}
        if algo == "eap-ig":
            extra_kw["ig_steps"] = 5
        elif algo == "ibcircuit":
            extra_kw["num_epochs"] = 1000

        pipe.discover(
            algorithm=algo,
            level=LEVEL,
            sparsity=SPARSITY,
            n_examples=N_EXAMPLES,
            batch_size=BATCH_SIZE,
            **extra_kw,
        )
        elapsed = time.time() - t0

        results[algo] = {
            "circuit": pipe._circuit,
            "time_s": elapsed,
            "pipe": pipe,
        }
        print(f"  ✓ {len(pipe._circuit)} nodes in {elapsed:.1f}s")
        print(f"  Top 5: {pipe._circuit.top_nodes(5)}")

    except Exception as e:
        elapsed = time.time() - t0
        print(f"  ✗ Failed after {elapsed:.1f}s: {e}")
        results[algo] = {"circuit": None, "time_s": elapsed, "error": str(e)}

    # Free GPU memory between algorithms
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"Completed {sum(1 for r in results.values() if r.get('circuit'))} / {len(ALGORITHMS)} algorithms")

## Summary Table

In [ ]:
rows = []
for algo, r in results.items():
    if r.get("circuit"):
        c = r["circuit"]
        top = list(c.top_nodes(3).keys())
        rows.append({
            "Algorithm": algo,
            "Nodes": len(c),
            "Time (s)": f"{r['time_s']:.1f}",
            "Top 3 Nodes": ", ".join(top),
        })
    else:
        rows.append({
            "Algorithm": algo,
            "Nodes": "FAILED",
            "Time (s)": f"{r['time_s']:.1f}",
            "Top 3 Nodes": r.get("error", "")[:60],
        })

df_summary = pd.DataFrame(rows)
df_summary

## Evaluate Each Circuit

Run patching + ablation on each discovered circuit to compare faithfulness.

In [ ]:
eval_results = {}

for algo, r in results.items():
    if r.get("circuit") is None:
        continue

    pipe = r["pipe"]
    print(f"Evaluating {algo}...")

    try:
        pipe.evaluate(
            pillars=["patching", "ablation"],
            n_examples=N_EXAMPLES,
        )
        eval_results[algo] = pipe._eval_report
        print(f"  ✓ Done")
    except Exception as e:
        import traceback
        print(f"  ✗ Eval failed: {e}")
        traceback.print_exc()
        eval_results[algo] = {"error": str(e)}

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Comparison Dashboard

Visualize score similarity across algorithms using the `ComparisonDashboard`.

In [ ]:
from circuitkit.visualize.comparison import ComparisonDashboard

# Build the circuits dict: {algo_name: {node_name: score}}
dashboard = None
score_dicts = {}
for algo, r in results.items():
    c = r.get("circuit")
    if c and c.scores:
        score_dicts[algo] = c.scores

if len(score_dicts) >= 2:
    dashboard = ComparisonDashboard(
        circuits=score_dicts,
        comparison_type="stability",
    )

    # Stability heatmap — how consistent are scores across algorithms?
    fig = dashboard.plot_stability_heatmap(top_k=20)
    fig.show()
else:
    print("Need at least 2 successful circuits for comparison.")

In [ ]:
# Correlation matrix — which algorithms agree?
if dashboard is not None:
    fig = dashboard.plot_correlation_matrix()
    fig.show()

In [ ]:
# Score distribution comparison
if dashboard is not None:
    fig = dashboard.plot_distribution_comparison()
    fig.show()

## Interpretation

**When to use which algorithm:**

- **EAP / EAP-IG** — Fast, reliable default. EAP-IG adds gradient integration for slightly higher quality at ~2× cost. Start here.
- **EAP-GP** — Gradient × path variant. Comparable quality to EAP-IG with different gradient attribution.
- **ACDC** — Iterative greedy search. Higher cost but finds sparser, more targeted circuits.
- **CD-T** — Causal discovery with statistical tests. **Node-level only** (no neuron-level). Good for well-defined causal structures.
- **IBCircuit** — Information bottleneck approach. **No paired data needed** — only clean inputs. Best when you can't construct meaningful corruptions.